# Cell-free Gator setup on Hamilton STAR

1. Choose a **platemap** (Twist CSV/Excel with Name, Well Location, Yield) and a **gatorsetup** Excel sheet.
2. Adjust volumes and deck locations if needed.
3. Click **Load sheets** to preview the worklist, **Setup deck**, then **Run**.

Default reaction: **16 µL cell-free mastermix + 4 µL DNA**. Dried DNA is resuspended to **40 ng/µL**. Water is aspirated with **cLLD just below the trough surface** so tips are not dunked (that was causing side droplets). Dispense is slow from above the well floor, with a gentle mix and a short settle. Only names in both the platemap and gatorsetup are run. DNA transfer uses **cLLD** and an off-center tip so it finds liquid instead of a bottom bubble.

Default deck (change in **Plate and carrier locations**):

- Rail 1, plate carrier: site 0 DNA plate, site 1 first Gator plate, site 2 second Gator plate
- Rail 7, tip carrier: 10 µL racks in sites 0–1, 300 µL racks in sites 3–4 (optional 50 µL in site 2)
- Rail 35, 32-position tube carrier: 2 mL mastermix tube
- Rail 43, trough carrier: site 0 water/TE

Mastermix is aspirated **one channel at a time** from the 2 mL tube using **conductivity liquid-level detection (cLLD)**, then dispensed in parallel. Each tip finds the surface and aspirates **2 mm below it**. The 2 mL tube is modeled **18 mm higher** than the 1.5 mL insert, and LLD cannot search closer than **5 mm** to the tube bottom. DNA transfer defaults to 10 µL tips; resuspend and mastermix default to 300 µL tips.


In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import io
import time
import traceback
from contextlib import redirect_stdout

import ipywidgets as widgets
from IPython.display import display

from cellfree_hamilton import (
    Settings,
    connect_liquid_handler,
    format_plan,
    load_run,
    resolve_file,
    run_protocol,
    setup_deck,
    validate_settings,
)

deck_is_ready = False
lh = None
current_plan = None
DEVELOPMENT_MODE = False

style = {"description_width": "210px"}
item_layout = widgets.Layout(width="440px")

title = widgets.HTML("<h3>Cell-free Gator setup</h3>")
instructions = widgets.HTML(
    "<b>Workflow:</b> Load Excel/CSV sheets, setup the STAR deck, then run. "
    "Names in <i>gatorsetup</i> are matched to the platemap Name column."
)

platemap_path = widgets.Text(
    value="platemap_pSHPs0831B0951017B_.csv",
    description="Platemap path",
    style=style,
    layout=item_layout,
)
platemap_upload = widgets.FileUpload(accept=".csv,.xlsx,.xls", multiple=False, description="Upload platemap")
gator_path = widgets.Text(
    value="gatorsetup.xlsx",
    description="Gatorsetup path",
    style=style,
    layout=item_layout,
)
gator_upload = widgets.FileUpload(accept=".xlsx,.xls,.csv", multiple=False, description="Upload gatorsetup")

dna_vol = widgets.BoundedFloatText(value=4.0, min=0.5, max=50, step=0.5, description="DNA (µL)", style=style, layout=item_layout)
mm_vol = widgets.BoundedFloatText(value=16.0, min=1, max=200, step=0.5, description="Mastermix (µL)", style=style, layout=item_layout)
target_conc = widgets.BoundedFloatText(value=40.0, min=0.1, max=1000, step=0.1, description="Target concentration", style=style, layout=item_layout)
conc_unit = widgets.Dropdown(options=["ng/uL", "nM"], value="ng/uL", description="Concentration unit", style=style, layout=item_layout)
min_resuspend = widgets.BoundedFloatText(value=5.0, min=2, max=200, step=1, description="Min resuspend (µL)", style=style, layout=item_layout)
max_resuspend = widgets.BoundedFloatText(value=120.0, min=10, max=300, step=1, description="Max resuspend (µL)", style=style, layout=item_layout)
resuspend_mix = widgets.BoundedIntText(value=5, min=0, max=20, description="Resuspend mix cycles", style=style, layout=item_layout)
dest_mix = widgets.BoundedIntText(value=3, min=0, max=20, description="Reaction mix cycles", style=style, layout=item_layout)
water_flow = widgets.BoundedFloatText(value=100.0, min=1, max=500, step=1, description="Water flow (µL/s)", style=style, layout=item_layout)
resuspend_dispense_flow = widgets.BoundedFloatText(value=30.0, min=1, max=200, step=1, description="Resuspend dispense (µL/s)", style=style, layout=item_layout)
resuspend_mix_flow = widgets.BoundedFloatText(value=30.0, min=1, max=200, step=1, description="Resuspend mix (µL/s)", style=style, layout=item_layout)
mm_flow = widgets.BoundedFloatText(value=50.0, min=1, max=500, step=1, description="Mastermix flow (µL/s)", style=style, layout=item_layout)
dna_flow = widgets.BoundedFloatText(value=40.0, min=1, max=500, step=1, description="DNA flow (µL/s)", style=style, layout=item_layout)
dna_lld = widgets.Checkbox(value=True, description="DNA transfer cLLD (skip bottom bubbles)")
water_lld = widgets.Checkbox(value=True, description="Water trough cLLD (avoid side droplets)")
mm_lld = widgets.Checkbox(value=True, description="Mastermix cLLD (conductivity)")
mm_immersion = widgets.BoundedFloatText(value=2.0, min=0.5, max=8, step=0.5, description="Immersion below surface (mm)", style=style, layout=item_layout)
mm_lld_sens = widgets.Dropdown(options=[("1 high", 1), ("2", 2), ("3", 3), ("4 low", 4)], value=2, description="cLLD sensitivity", style=style, layout=item_layout)
mm_tube_z = widgets.BoundedFloatText(value=18.0, min=0, max=40, step=1, description="2 mL tube Z raise (mm)", style=style, layout=item_layout)
mm_min_z = widgets.BoundedFloatText(value=5.0, min=2, max=20, step=0.5, description="Min height above bottom (mm)", style=style, layout=item_layout)

do_resuspend = widgets.Checkbox(value=True, description="Resuspend DNA")
do_mastermix = widgets.Checkbox(value=True, description="Dispense mastermix")
do_dna_transfer = widgets.Checkbox(value=True, description="Transfer DNA to Gator plates")
simulation_toggle = widgets.Checkbox(value=False, description="Simulation (no robot)")
development_toggle = widgets.Checkbox(value=False, description="Development mode (verbose errors)")

tip_rail = widgets.BoundedIntText(value=7, min=-20, max=60, description="Tip carrier rail", style=style, layout=item_layout)
plate_rail = widgets.BoundedIntText(value=1, min=-20, max=60, description="Plate carrier rail", style=style, layout=item_layout)
trough_rail = widgets.BoundedIntText(value=43, min=-20, max=60, description="Trough carrier rail", style=style, layout=item_layout)
tube_rail = widgets.BoundedIntText(value=35, min=-20, max=60, description="Tube carrier rail", style=style, layout=item_layout)
dna_site = widgets.BoundedIntText(value=0, min=0, max=4, description="DNA plate site", style=style, layout=item_layout)
gator1_site = widgets.BoundedIntText(value=1, min=0, max=4, description="Gator plate 1 site", style=style, layout=item_layout)
gator2_site = widgets.BoundedIntText(value=2, min=0, max=4, description="Gator plate 2 site", style=style, layout=item_layout)
water_site = widgets.BoundedIntText(value=0, min=0, max=3, description="Water trough site", style=style, layout=item_layout)
mm_tube_site = widgets.BoundedIntText(value=0, min=0, max=31, description="Mastermix 2 mL site", style=style, layout=item_layout)
load_10 = widgets.Checkbox(value=True, description="Load 10 µL tips")
load_50 = widgets.Checkbox(value=False, description="Load 50 µL tips")
load_300 = widgets.Checkbox(value=True, description="Load 300 µL tips")
site_10 = widgets.BoundedIntText(value=0, min=0, max=4, description="10 µL rack site", style=style, layout=item_layout)
site_10_extra = widgets.BoundedIntText(value=1, min=-1, max=4, description="10 µL extra site (-1 none)", style=style, layout=item_layout)
site_50 = widgets.BoundedIntText(value=2, min=0, max=4, description="50 µL rack site", style=style, layout=item_layout)
site_50_extra = widgets.BoundedIntText(value=-1, min=-1, max=4, description="50 µL extra site (-1 none)", style=style, layout=item_layout)
site_300 = widgets.BoundedIntText(value=3, min=0, max=4, description="300 µL rack site", style=style, layout=item_layout)
site_300_extra = widgets.BoundedIntText(value=4, min=-1, max=4, description="300 µL extra site (-1 none)", style=style, layout=item_layout)
resuspend_tip_size = widgets.Dropdown(options=[10, 50, 300], value=300, description="Resuspend tip (µL)", style=style, layout=item_layout)
mastermix_tip_size = widgets.Dropdown(options=[10, 50, 300], value=300, description="Mastermix tip (µL)", style=style, layout=item_layout)
dna_tip_size = widgets.Dropdown(options=[10, 50, 300], value=10, description="DNA transfer tip (µL)", style=style, layout=item_layout)

load_button = widgets.Button(description="Load sheets", button_style="primary")
setup_button = widgets.Button(description="Setup deck", button_style="info")
run_button = widgets.Button(description="Run", button_style="success", disabled=True)
return_tips_button = widgets.Button(description="Return 8-channel tips", button_style="warning", disabled=True)
trash_tips_button = widgets.Button(description="Trash 8-channel tips", button_style="danger", disabled=True)

status_output = widgets.Output(layout={"border": "1px solid #ddd", "padding": "8px", "max_height": "360px", "overflow": "auto"})
run_state_label = widgets.HTML("<b>Status:</b> Idle")
deck_summary = widgets.Textarea(value="", description="Deck layout", layout={"width": "100%", "height": "220px"}, disabled=True)
plan_preview = widgets.Textarea(value="", description="Worklist", layout={"width": "100%", "height": "280px"}, disabled=True)
reagent_note = widgets.HTML("")


def _collect_settings() -> Settings:
    settings = Settings(
        dna_vol_ul=float(dna_vol.value),
        mastermix_vol_ul=float(mm_vol.value),
        target_concentration=float(target_conc.value),
        concentration_unit=str(conc_unit.value),
        min_resuspend_ul=float(min_resuspend.value),
        max_resuspend_ul=float(max_resuspend.value),
        resuspend_mix_cycles=int(resuspend_mix.value),
        dest_mix_cycles=int(dest_mix.value),
        water_flow_rate=float(water_flow.value),
        water_lld=bool(water_lld.value),
        resuspend_dispense_flow_rate=float(resuspend_dispense_flow.value),
        resuspend_mix_flow_rate=float(resuspend_mix_flow.value),
        mastermix_flow_rate=float(mm_flow.value),
        mastermix_lld=bool(mm_lld.value),
        mastermix_immersion_mm=float(mm_immersion.value),
        mastermix_lld_sensitivity=int(mm_lld_sens.value),
        mastermix_tube_z_offset_mm=float(mm_tube_z.value),
        mastermix_min_height_mm=float(mm_min_z.value),
        dna_flow_rate=float(dna_flow.value),
        dna_lld=bool(dna_lld.value),
        do_resuspend=bool(do_resuspend.value),
        do_mastermix=bool(do_mastermix.value),
        do_dna_transfer=bool(do_dna_transfer.value),
        simulation=bool(simulation_toggle.value),
        tip_carrier_rail=int(tip_rail.value),
        plate_carrier_rail=int(plate_rail.value),
        trough_carrier_rail=int(trough_rail.value),
        tube_carrier_rail=int(tube_rail.value),
        dna_plate_site=int(dna_site.value),
        gator_plate_1_site=int(gator1_site.value),
        gator_plate_2_site=int(gator2_site.value),
        water_trough_site=int(water_site.value),
        mastermix_tube_site=int(mm_tube_site.value),
        load_10ul_tips=bool(load_10.value),
        load_50ul_tips=bool(load_50.value),
        load_300ul_tips=bool(load_300.value),
        tips_10ul_site=int(site_10.value),
        tips_10ul_extra_site=int(site_10_extra.value),
        tips_50ul_site=int(site_50.value),
        tips_50ul_extra_site=int(site_50_extra.value),
        tips_300ul_site=int(site_300.value),
        tips_300ul_extra_site=int(site_300_extra.value),
        resuspend_tip_ul=int(resuspend_tip_size.value),
        mastermix_tip_ul=int(mastermix_tip_size.value),
        dna_tip_ul=int(dna_tip_size.value),
    )
    validate_settings(settings)
    return settings


def _log(message: str) -> None:
    status_output.append_stdout(message + ("" if message.endswith("\n") else "\n"))


def _load_plan():
    global current_plan, DEVELOPMENT_MODE
    DEVELOPMENT_MODE = bool(development_toggle.value)
    settings = _collect_settings()
    platemap_bytes, platemap_name = resolve_file(platemap_upload.value, platemap_path.value)
    gator_bytes, gator_name = resolve_file(gator_upload.value, gator_path.value)
    current_plan = load_run(platemap_bytes, gator_bytes, settings, platemap_name=platemap_name)
    plan_preview.value = format_plan(current_plan)
    titles = current_plan.gator_titles
    gator_labels = " and ".join(titles) if titles else "two Gator plates"
    reagent_note.value = (
        f"<b>Load:</b> {current_plan.water_ul:.0f} µL water (+extra) in the water trough; "
        f"{current_plan.mastermix_ul:.0f} µL cell-free mastermix in the 2 mL tube. "
        f"Gator plates: {gator_labels}. Files: {platemap_name}, {gator_name}."
    )
    return current_plan


async def _setup_deck_async():
    global lh, deck_is_ready, DEVELOPMENT_MODE
    setup_button.disabled = True
    run_button.disabled = True
    return_tips_button.disabled = True
    trash_tips_button.disabled = True
    DEVELOPMENT_MODE = bool(development_toggle.value)
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    _log("Setting up deck...")
    try:
        if deck_is_ready:
            raise RuntimeError("Deck is already set up. Restart the kernel to change hardware layout.")
        settings = _collect_settings()
        lh = setup_deck(settings)
        buf = io.StringIO()
        with redirect_stdout(buf):
            summary = await connect_liquid_handler(lh, settings)
        deck_is_ready = True
        deck_summary.value = (buf.getvalue().strip() + "\n" + str(summary)).strip()
        _log("Deck setup complete.")
        run_button.disabled = current_plan is None
        return_tips_button.disabled = False
        trash_tips_button.disabled = False
    except Exception as exc:
        _log(f"Setup failed: {exc}")
        if DEVELOPMENT_MODE:
            _log(traceback.format_exc())
    finally:
        setup_button.disabled = False


async def _run_async():
    global current_plan, DEVELOPMENT_MODE
    setup_button.disabled = True
    run_button.disabled = True
    return_tips_button.disabled = True
    trash_tips_button.disabled = True
    DEVELOPMENT_MODE = bool(development_toggle.value)
    run_state_label.value = "<b>Status:</b> Running..."
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    _log("Starting run...")
    _log(time.ctime())
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup deck first.")
        current_plan = _load_plan()
        _log(f"Running {len(current_plan.transfers)} reactions on {', '.join(current_plan.gator_titles)}.")
        buf = io.StringIO()
        with redirect_stdout(buf):
            await run_protocol(lh, current_plan, log=_log)
        chatter = buf.getvalue().strip()
        if chatter:
            _log(chatter)
        _log("Run complete.")
        _log(time.ctime())
    except Exception as exc:
        _log(f"Run failed: {exc}")
        if DEVELOPMENT_MODE:
            _log(traceback.format_exc())
    finally:
        setup_button.disabled = False
        run_button.disabled = False
        return_tips_button.disabled = False
        trash_tips_button.disabled = False
        run_state_label.value = "<b>Status:</b> Idle"


async def _return_tips_async():
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup deck first.")
        _log("Returning 8-channel tips...")
        await lh.return_tips(allow_nonzero_volume=True)
        _log("Returned 8-channel tips.")
    except Exception as exc:
        _log(f"Return tips failed: {exc}")
        if development_toggle.value:
            _log(traceback.format_exc())


async def _trash_tips_async():
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup deck first.")
        _log("Trashing 8-channel tips...")
        await lh.discard_tips(allow_nonzero_volume=True)
        _log("Trashed 8-channel tips.")
    except Exception as exc:
        _log(f"Trash tips failed: {exc}")
        if development_toggle.value:
            _log(traceback.format_exc())


def _on_load(_):
    try:
        if not development_toggle.value:
            status_output.clear_output()
        _load_plan()
        skipped = len(current_plan.unused_dna_names) + len(current_plan.unmatched_dest_names)
        _log(
            f"Loaded {len(current_plan.transfers)} reactions for names shared by both sheets."
            + (f" Skipped {skipped} names that appear on only one sheet." if skipped else "")
        )
        if deck_is_ready:
            run_button.disabled = False
    except Exception as exc:
        _log(f"Load failed: {exc}")
        if development_toggle.value:
            _log(traceback.format_exc())


def _on_setup(_):
    asyncio.create_task(_setup_deck_async())


def _on_run(_):
    asyncio.create_task(_run_async())


def _on_return(_):
    asyncio.create_task(_return_tips_async())


def _on_trash(_):
    asyncio.create_task(_trash_tips_async())


load_button.on_click(_on_load)
setup_button.on_click(_on_setup)
run_button.on_click(_on_run)
return_tips_button.on_click(_on_return)
trash_tips_button.on_click(_on_trash)

advanced = widgets.Accordion(
    children=[
        widgets.VBox([dna_vol, mm_vol, target_conc, conc_unit, min_resuspend, max_resuspend, resuspend_mix, dest_mix, water_flow, resuspend_dispense_flow, resuspend_mix_flow, mm_flow, dna_flow, water_lld, dna_lld, mm_lld, mm_immersion, mm_lld_sens, mm_tube_z, mm_min_z]),
        widgets.VBox([
            load_10, load_50, load_300,
            resuspend_tip_size, mastermix_tip_size, dna_tip_size,
            site_10, site_10_extra, site_50, site_50_extra, site_300, site_300_extra,
        ]),
        widgets.VBox([
            tip_rail, plate_rail, trough_rail, tube_rail,
            dna_site, gator1_site, gator2_site,
            water_site, mm_tube_site,
        ]),
    ]
)
advanced.set_title(0, "Volumes and mixing")
advanced.set_title(1, "Tip sizes")
advanced.set_title(2, "Plate and carrier locations")

controls = widgets.VBox([
    title,
    instructions,
    widgets.HTML("<b>Files</b>"),
    platemap_path,
    platemap_upload,
    gator_path,
    gator_upload,
    widgets.HTML("<b>Steps</b>"),
    do_resuspend,
    do_mastermix,
    do_dna_transfer,
    simulation_toggle,
    development_toggle,
    widgets.HTML("<b>Advanced</b>"),
    advanced,
    widgets.HBox([load_button, setup_button, run_button]),
    widgets.HTML("<b>Tip recovery</b>"),
    widgets.HBox([return_tips_button, trash_tips_button]),
    reagent_note,
    plan_preview,
    widgets.HTML("<b>Loaded deck</b>"),
    deck_summary,
    run_state_label,
    status_output,
])

display(controls)